# EEG State Classification: Focus vs. Relaxation

In this notebook you will work through a complete EEG analysis pipeline — from raw data to trained classifiers. The data comes from a simple two-class experiment:

| Condition | Instruction | Duration |
|-----------|-------------|----------|
| **Focus** | Repeatedly subtract 7 from a 3-digit number | 10 s |
| **Relaxation** | Relax and breathe naturally | 10 s |

Between trials there is a 4 s fixation/blink period (unlabeled).

**Recording setup:** 7-channel Exon EEG headset, XDF format with LSL markers.

### What you will do
1. Load the XDF recording and build an MNE `Raw` object
2. Preprocess: bandpass filter, notch filter, re-reference, artifact rejection
3. Epoch the continuous data using LSL markers
4. **Analysis A** — Compare alpha-band power between the two states
5. **Analysis B** — Train LDA and SVM classifiers on spectral features; evaluate on a held-out causal test set

Cells marked with **`# TODO`** require your implementation. Helper code and plotting are provided so you can focus on the signal-processing and ML concepts.

> **Time estimate:** ~60–90 min

## 0 — Environment Setup

Run this cell once to install the packages you need. If you already have them, it will finish quickly.

In [ ]:
%pip install pyxdf mne scikit-learn matplotlib numpy scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import pyxdf
from pathlib import Path
from scipy import stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

mne.set_log_level("WARNING")
%matplotlib inline

---
## 1 — Load the XDF Recording

[XDF](https://github.com/sccn/xdf) is the standard file format for Lab Streaming Layer (LSL) recordings. A single `.xdf` file bundles **multiple time-series streams** — in our case one EEG data stream and one event-marker stream — each with its own clock that `pyxdf` aligns for us.

The cell below loads the file and separates the two streams. You don't need to modify it, but **read through it carefully** — understanding what `streams` contains is essential for everything that follows.

In [ ]:
XDF_PATH = "sub-P001_ses-S001_task-Default_run-001_eeg.xdf"

streams, header = pyxdf.load_xdf(XDF_PATH)

# Separate EEG and marker streams by their declared type
eeg_stream = None
marker_stream = None

for s in streams:
    stream_type = s["info"]["type"][0].lower()
    if stream_type in ("eeg", "data"):
        eeg_stream = s
    elif stream_type in ("markers", "marker", "events"):
        marker_stream = s

# Fallback: if type metadata is ambiguous, use channel count heuristic
if eeg_stream is None or marker_stream is None:
    for s in streams:
        n_ch = int(s["info"]["channel_count"][0])
        if n_ch > 1 and eeg_stream is None:
            eeg_stream = s
        elif n_ch == 1 and marker_stream is None:
            marker_stream = s

assert eeg_stream is not None, "Could not find EEG stream in XDF file"
assert marker_stream is not None, "Could not find marker stream in XDF file"

sfreq = float(eeg_stream["info"]["nominal_srate"][0])
n_recorded_channels = int(eeg_stream["info"]["channel_count"][0])
eeg_data = eeg_stream["time_series"]        # shape: (n_samples, n_channels)
eeg_times = eeg_stream["time_stamps"]        # shape: (n_samples,)
marker_strings = [m[0] for m in marker_stream["time_series"]]
marker_times = marker_stream["time_stamps"]

print(f"EEG stream : {n_recorded_channels} recorded channels, {sfreq} Hz, {len(eeg_times)} samples "
      f"({len(eeg_times)/sfreq:.1f} s)")
print(f"Markers    : {len(marker_strings)} events")
print(f"Unique markers: {sorted(set(marker_strings))}")

### 1.1 — Build an MNE `Raw` object

MNE-Python is the standard library for EEG/MEG analysis. Its core data structure is `Raw` — a container that pairs a (channels × samples) data matrix with metadata like channel names, sampling rate, and a measurement info dict.

We also need to inject the LSL markers as MNE **annotations**, which are `(onset, duration, description)` triples anchored to the Raw timeline. These annotations will later let us cut the continuous recording into labeled **epochs**.

In [ ]:
# Keep only the 7 scalp EEG channels used in the experiment.
# The XDF stream also contains BIP and accelerometer channels.
SCALP_CHANNELS = ["F3", "F4", "C3", "Cz", "C4", "P3", "P4"]

try:
    ch_info = eeg_stream["info"]["desc"][0]["channels"][0]["channel"]
    all_ch_names = [ch["label"][0] for ch in ch_info]
except (KeyError, TypeError, IndexError):
    all_ch_names = [f"CH{i+1}" for i in range(n_recorded_channels)]

channel_lookup = {name: idx for idx, name in enumerate(all_ch_names)}
missing = [name for name in SCALP_CHANNELS if name not in channel_lookup]
assert not missing, f"Missing expected EEG channels: {missing}"

eeg_indices = [channel_lookup[name] for name in SCALP_CHANNELS]
ch_names = [all_ch_names[idx] for idx in eeg_indices]
scalp_eeg = eeg_data[:, eeg_indices]

print(f"Using scalp EEG channels: {ch_names}")

# MNE expects data in Volts. Some XDF exports label EEG as volts even when
# the stored values are actually in microvolts, so we infer the scale from
# the observed amplitudes.
scale_hint = np.nanpercentile(np.abs(scalp_eeg), 99)
scale = 1e-6 if scale_hint > 1e-3 else 1.0
scale_label = "uV -> V" if scale == 1e-6 else "already V"
print(f"Applying EEG scale: {scale_label} (99th percentile abs amplitude = {scale_hint:.3g})")

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=["eeg"] * len(ch_names))
raw = mne.io.RawArray((scalp_eeg * scale).T, info)

# Convert LSL marker timestamps to seconds relative to the Raw start
raw_start = eeg_times[0]
onsets = marker_times - raw_start

# Keep only *_start markers — these mark the beginning of each trial
start_mask = [m.endswith("_start") for m in marker_strings]
annot_onsets = onsets[start_mask]
annot_descriptions = [m for m, keep in zip(marker_strings, start_mask) if keep]
annot_durations = [0.0] * len(annot_onsets)

annotations = mne.Annotations(onset=annot_onsets,
                               duration=annot_durations,
                               description=annot_descriptions)
raw.set_annotations(annotations)

print(f"\nRaw object : {raw.info['nchan']} channels, {raw.n_times} samples, "
      f"{raw.times[-1]:.1f} s")
print(f"Annotations: {len(raw.annotations)} trial-start markers")
raw.plot(duration=30, n_channels=len(ch_names), scalings="auto", title="Raw EEG (first 30 s)")
plt.show()

---
## 2 — Preprocessing

Raw EEG is dominated by noise: line noise from the power grid, slow drifts from electrode impedance changes, muscle artifacts, and eye blinks. Preprocessing removes or attenuates these so that the neural signal of interest can emerge.

The standard minimal pipeline is:

1. **Bandpass filter** (e.g. 1–40 Hz) — removes slow drifts (< 1 Hz) and high-frequency muscle noise (> 40 Hz)
2. **Notch filter** at the power-line frequency (60 Hz in North America) — removes electrical interference
3. **Re-reference** — EEG measures *voltage differences*. Choosing a reference (e.g. the average of all channels) standardizes what "zero" means across channels.

**Why does the order matter?** Filtering before re-referencing avoids spreading line noise or drift from one noisy channel into all channels via the average.

### 2.1 — Bandpass filter

A bandpass filter keeps frequencies within a range and attenuates everything outside it.

**Intuition:** Think of it as a window on the frequency axis. Neural oscillations we care about (theta 4–8 Hz, alpha 8–13 Hz, beta 13–30 Hz) all live between 1 and 40 Hz. Anything outside is almost certainly noise for our purposes.

In [ ]:
# TODO: Apply a bandpass filter to `raw` keeping frequencies between 1 and 40 Hz.
#
# Hint: MNE Raw objects have a `.filter()` method.
#       It modifies the object in-place and returns it.
#       You need to pass the low and high cutoff frequencies.
#       Then plot the filtered raw EEG so you can inspect the result.
#
# raw.filter(l_freq=???, h_freq=???)
# raw.plot(duration=10, n_channels=len(ch_names), scalings="auto",
#          title="Raw EEG after 1-40 Hz bandpass")
# plt.show()



### 2.2 — Notch filter

Even after bandpassing, residual eletrical line noise can survive (especially if your bandpass upper edge is above 50/60 Hz, or if the noise is strong). A notch filter surgically removes a narrow band around a target frequency - 50hz for Europe and 60hz for America.

**Intuition:** A notch filter is like covering one specific key on a piano — it silences exactly that frequency while leaving its neighbors untouched.

In [ ]:
# TODO: Apply a notch filter at 60 Hz to remove power-line noise.
#
# Hint: MNE Raw objects have a `.notch_filter()` method.
#       Pass the frequency (or list of frequencies) to notch out.
#       Then plot the raw EEG again to compare against the bandpassed signal.
#
# raw.notch_filter(freqs=???)
# raw.plot(duration=10, n_channels=len(ch_names), scalings="auto",
#          title="Raw EEG after 60 Hz notch")
# plt.show()



### 2.3 — Re-referencing

EEG electrodes don't measure absolute voltages — they measure the *difference* between each electrode and a **reference** electrode. The choice of reference changes the shape of the signal.

**Average reference** subtracts the mean across all channels at each time point. This is a common choice when you don't have a dedicated reference electrode, and it works well when you have reasonable spatial coverage.

**Intuition:** Imagine several thermometers in a room. If one reads 22 °C and the room average is 20 °C, its average-referenced value is +2 °C. You lose the absolute temperature but gain a clearer picture of *which spots are warmer or cooler than typical*.

In [ ]:
# TODO: Set the EEG reference to the average of all channels.
#
# Hint: Use `raw.set_eeg_reference()`. The default argument for
#       ref_channels is "average", which is what you want here.
#       Then plot the re-referenced raw EEG one more time.
#
# raw.set_eeg_reference(???)
# raw.plot(duration=10, n_channels=len(ch_names), scalings="auto",
#          title="Raw EEG after average reference")
# plt.show()



### 2.4 — Verify preprocessing

Let's visualize the PSD (power spectral density) of the preprocessed data. You should see:
- A clear drop-off above 40 Hz (bandpass)
- A notch (dip) at 60 Hz if there was any line noise leakage
- The bulk of the power concentrated in the 1–30 Hz range

In [ ]:
raw.compute_psd(fmax=80).plot()
plt.suptitle("PSD after preprocessing", y=1.02)
plt.show()

---
## 3 — Epoching

So far we have one long continuous recording. To do any trial-level analysis we need to **cut it into chunks (epochs)** — one per trial, time-locked to the marker events.

An **epoch** is a short segment of EEG extracted around an event of interest. Here each trial lasts 10 s, so we'll extract 0–10 s after each `*_start` marker.

We'll also apply a simple **amplitude-based artifact rejection**: if any channel in an epoch exceeds a peak-to-peak threshold, the whole epoch is dropped. This removes trials contaminated by large muscle artifacts or electrode pops.

**Intuition for the threshold:** Clean EEG rarely exceeds ~100 µV peak-to-peak. Anything much larger is almost certainly an artifact. For this dry-electrode recording, we use 300 µV as a looser threshold so we still keep enough usable trials.

In [ ]:
# TODO: Create MNE Epochs from the annotated Raw object.
#
# Steps:
#   1. Use `mne.events_from_annotations()` to convert annotations → events array
#      and get the event_id mapping dict.
#   2. Use `mne.Epochs()` to cut the data. Key parameters:
#        - tmin=0.0, tmax=10.0 - 1.0 / sfreq  (exactly 10 s at this sample rate)
#        - baseline=None         (no baseline correction — our trials aren't
#                                 time-locked to a brief stimulus)
#        - reject=dict(eeg=300e-6)  (drop epochs where any channel > 300 µV)
#        - preload=True          (load data into memory so we can work with it)
#
# events, event_id = mne.events_from_annotations(raw)
# epochs = mne.Epochs(raw, events, event_id, tmin=???, tmax=???,
#                     baseline=???, reject=???, preload=???)



In [ ]:
# Inspect what we got
print(f"Event ID mapping: {event_id}")
print(epochs)
print(f"\nEpochs kept per condition:")
counts = {cond: len(epochs[cond]) for cond in event_id}
for cond, count in counts.items():
    print(f"  {cond}: {count}")

empty_conditions = [cond for cond, count in counts.items() if count == 0]
if empty_conditions:
    raise RuntimeError(
        "No epochs left for one or more conditions after artifact rejection: "
        f"{empty_conditions}. Rerun the epoching cell with a larger reject threshold or reject=None."
    )

---
## 4 — Analysis A: Alpha Power

Alpha oscillations (8–13 Hz) are one of the most robust EEG signatures. They are typically **strongest during relaxed, eyes-closed states** and **suppressed (desynchronized) during active mental effort**.

**Prediction:** If the experiment worked, we should see *higher* alpha power during relaxation trials and *lower* alpha power during focus (mental arithmetic) trials.

We'll test this by:
1. Computing the power spectral density (PSD) of each epoch
2. Extracting the mean power in the alpha band (8–13 Hz)
3. Comparing distributions with a statistical test and a plot

### 4.1 — Compute alpha power per epoch

In [ ]:
ALPHA_LOW, ALPHA_HIGH = 8.0, 13.0

def compute_band_power(epochs_obj, fmin, fmax):
    """Compute mean power in a frequency band for each epoch.

    Returns an array of shape (n_epochs,) — one scalar per epoch,
    averaged across all channels.
    """
    # TODO: Compute the PSD for each epoch, then extract and average the
    #       alpha-band power.
    #
    # Steps:
    #   1. Check that there is at least one epoch left for this condition.
    #   2. Call epochs_obj.compute_psd(method="welch", fmin=fmin, fmax=fmax)
    #      to get a Spectrum object restricted to the band of interest.
    #   3. Use .get_data() on the Spectrum to get the raw power array
    #      with shape (n_epochs, n_channels, n_freqs).
    #   4. Average over channels (axis=1) and frequencies (axis=2) to get
    #      one value per epoch → shape (n_epochs,).
    #
    # if len(epochs_obj) == 0:
    #     raise RuntimeError("No epochs available for this condition.")
    # psd = epochs_obj.compute_psd(method="welch", fmin=???, fmax=???)
    # power = psd.get_data()          # shape: (n_epochs, n_channels, n_freqs)
    # return power.mean(axis=???).mean(axis=???)
    pass


alpha_focus = compute_band_power(epochs["focus_start"], ALPHA_LOW, ALPHA_HIGH)
alpha_relax = compute_band_power(epochs["relaxation_start"], ALPHA_LOW, ALPHA_HIGH)

print(f"Focus alpha power     : mean={alpha_focus.mean():.4e}, std={alpha_focus.std():.4e}")
print(f"Relaxation alpha power: mean={alpha_relax.mean():.4e}, std={alpha_relax.std():.4e}")

### 4.2 — Visualize and test

We'll make a box plot to eyeball the difference, and run a two-sample t-test to quantify it.

**Question to think about:** Why might a t-test be imperfect here? (Hint: are consecutive trials truly independent? What would a permutation test buy you?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
axes[0].boxplot([alpha_focus, alpha_relax], labels=["Focus", "Relaxation"])
axes[0].set_ylabel("Alpha power (V²/Hz)")
axes[0].set_title("Alpha Power by Condition")

# Overlay individual data points for transparency
for i, (data, color) in enumerate([(alpha_focus, "steelblue"), (alpha_relax, "salmon")]):
    x = np.random.normal(i + 1, 0.04, size=len(data))
    axes[0].scatter(x, data, alpha=0.4, s=15, color=color)

# Histogram / distribution overlap
axes[1].hist(alpha_focus, bins=15, alpha=0.5, label="Focus", color="steelblue")
axes[1].hist(alpha_relax, bins=15, alpha=0.5, label="Relaxation", color="salmon")
axes[1].set_xlabel("Alpha power (V²/Hz)")
axes[1].set_ylabel("Count")
axes[1].set_title("Alpha Power Distributions")
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistical test
t_stat, p_value = stats.ttest_ind(alpha_focus, alpha_relax)
print(f"Two-sample t-test:  t = {t_stat:.3f},  p = {p_value:.4f}")
if p_value < 0.05:
    print("→ Significant difference in alpha power between conditions (p < 0.05)")
else:
    print("→ No significant difference detected (p ≥ 0.05)")

---
## 5 — Analysis B: ML Classification

Now we move from a single-feature comparison to a full classification pipeline. The goal: given an epoch of EEG, predict whether the participant was in the **focus** or **relaxation** state.

### Feature design

For each epoch we'll extract **band power** in four standard frequency bands across all channels:

| Band | Range | Associated with |
|------|-------|----------------|
| Delta | 1–4 Hz | Deep sleep, slow artifacts |
| Theta | 4–8 Hz | Drowsiness, memory encoding |
| Alpha | 8–13 Hz | Relaxed wakefulness, idling |
| Beta | 13–30 Hz | Active thinking, motor planning |

This gives us `4 bands × n_channels` features per epoch — a compact, interpretable feature vector.

### Train / test split strategy

We use a **causal split**: the first 80% of trials (in recording order) form the training set, the last 20% form the test set. This simulates a realistic scenario where you train on past data and predict future states — no peeking into the future.

**Why not random split or cross-validation?** EEG signals have slow autocorrelation — trials close in time are more similar than trials far apart. A random split would leak temporal information and inflate accuracy. The causal split gives an honest estimate of how well the classifier would work on *new, future* data.

### 5.1 — Extract features

In [ ]:
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
}

def extract_band_features(epochs_obj):
    """Extract band-power features for each epoch.

    Returns:
        X : np.ndarray of shape (n_epochs, n_channels * n_bands)
            Feature matrix — each row is one epoch's feature vector.
        feature_names : list of str
            Human-readable name for each feature column.
    """
    # TODO: For each frequency band, compute the PSD restricted to that band,
    #       then average over frequencies to get power per channel per epoch.
    #       Concatenate across bands to form the feature matrix.
    #
    # Skeleton:
    #   all_band_powers = []
    #   feature_names = []
    #   for band_name, (fmin, fmax) in BANDS.items():
    #       psd = epochs_obj.compute_psd(fmin=???, fmax=???)
    #       power = psd.get_data()                        # (n_epochs, n_channels, n_freqs)
    #       band_power = power.mean(axis=???)              # (n_epochs, n_channels) — avg over freqs
    #       all_band_powers.append(band_power)
    #       for ch in epochs_obj.ch_names:
    #           feature_names.append(f"{ch}_{band_name}")
    #   X = np.concatenate(all_band_powers, axis=???)      # (n_epochs, n_channels * n_bands)
    #   return X, feature_names
    pass


X, feature_names = extract_band_features(epochs)
print(f"Feature matrix shape: {X.shape}  ({X.shape[0]} epochs × {X.shape[1]} features)")
print(f"Features: {feature_names}")

### 5.2 — Build labels and causal train/test split

In [ ]:
# Build label vector: 0 = focus, 1 = relaxation
# epochs.events[:, 2] contains the integer event codes from MNE.
# We need to map them back to our condition names.
id_to_name = {v: k for k, v in event_id.items()}
labels = np.array([1 if "relax" in id_to_name[code] else 0
                   for code in epochs.events[:, 2]])

label_names = {0: "focus", 1: "relaxation"}
print(f"Labels: {np.bincount(labels)}  (0=focus, 1=relaxation)")

# TODO: Split into train (first 80%) and test (last 20%) sets.
#       Remember — this is a *causal* split, so we split by time order,
#       NOT randomly.
#
# Hint: compute the split index, then slice X and labels.
#
# split_idx = int(len(labels) * ???)
# X_train, X_test = X[:split_idx], X[split_idx:]
# y_train, y_test = labels[:split_idx], labels[split_idx:]


print(f"\nTrain: {len(y_train)} epochs  |  Test: {len(y_test)} epochs")
print(f"Train class balance: focus={np.sum(y_train==0)}, relaxation={np.sum(y_train==1)}")
print(f"Test  class balance: focus={np.sum(y_test==0)}, relaxation={np.sum(y_test==1)}")

### 5.3 — Train classifiers

We'll train two classifiers and compare them:

**LDA (Linear Discriminant Analysis)** finds a single linear boundary that maximally separates the two classes. It's fast, has very few parameters, and works surprisingly well on EEG band-power features. It's the go-to baseline in BCI research.

**SVM with RBF kernel (Support Vector Machine)** finds a nonlinear decision boundary by mapping features into a higher-dimensional space. It can capture interactions between features that LDA misses, but is more prone to overfitting on small datasets.

Both classifiers are wrapped in a `Pipeline` with `StandardScaler` so that features are z-scored before fitting. This is important because band powers across different frequency ranges can differ by orders of magnitude — without scaling, features with larger variance would dominate.

**Question to think about:** With only ~80 training epochs and 28 features, which classifier do you expect to perform better? Why?

In [ ]:
# TODO: Create and train two classifier pipelines.
#
# For each classifier:
#   1. Build a sklearn Pipeline with StandardScaler + the classifier
#   2. Fit it on (X_train, y_train)
#
# Hints:
#   - make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
#   - make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
#   - Call .fit(X_train, y_train) on each pipeline
#
# lda_pipe = make_pipeline(???, ???)
# lda_pipe.fit(???, ???)
#
# svm_pipe = make_pipeline(???, ???)
# svm_pipe.fit(???, ???)



### 5.4 — Evaluate on the test set

In [ ]:
classifiers = {"LDA": lda_pipe, "SVM (RBF)": svm_pipe}

fig, axes = plt.subplots(1, len(classifiers), figsize=(5 * len(classifiers), 4))
if len(classifiers) == 1:
    axes = [axes]

for ax, (name, clf) in zip(axes, classifiers.items()):
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"\n{'='*40}")
    print(f"{name}  —  Test accuracy: {acc:.1%}")
    print(f"{'='*40}")
    print(classification_report(y_test, y_pred,
                                target_names=["focus", "relaxation"]))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["focus", "relaxation"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nAccuracy: {acc:.1%}")

plt.tight_layout()
plt.show()

### 5.5 — Feature importance

Which features mattered most? LDA gives us a direct answer through its **coefficients** — larger absolute weight means the feature contributed more to the decision boundary.

In [ ]:
# Extract LDA coefficients (the scaler doesn't change relative ordering much,
# but we use the raw coefs from the LDA step for interpretability)
lda_model = lda_pipe.named_steps["lineardiscriminantanalysis"]
coefs = lda_model.coef_.ravel()

sorted_idx = np.argsort(np.abs(coefs))[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(coefs)), coefs[sorted_idx], color="steelblue")
ax.set_yticks(range(len(coefs)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("LDA coefficient")
ax.set_title("Feature importance (LDA weights)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 features by |weight|:")
for rank, idx in enumerate(sorted_idx[:5], 1):
    print(f"  {rank}. {feature_names[idx]:20s}  weight = {coefs[idx]:+.4f}")

---
## 6 — Discussion

Now that you've run the full pipeline, reflect on these questions:

1. **Alpha power result:** Was relaxation alpha power higher than focus alpha power, as predicted? If not, what could explain the discrepancy? (Hint: eyes open vs. closed, participant compliance, electrode placement over occipital cortex)

2. **Classifier performance:** How did LDA and SVM compare? If one was clearly better, why might that be given the dataset size and feature dimensionality?

3. **Feature importance:** Did the alpha-band features dominate the LDA weights, or were other bands (theta, beta) also informative? What does this tell you about the neural signatures of mental arithmetic?

4. **Causal split:** Look at the test set class balance. Is it roughly 50/50? If not, how might that affect your interpretation of accuracy? (Hint: what would a "dummy" classifier that always predicts the majority class achieve?)

5. **Limitations:** This is a single subject, single session. What would you need to do to build a classifier that generalizes across people? What about across days for the same person?

### Going further (optional challenges)
- Try adding **log-transformed** band power as features. Why might log-scaling help?
- Implement a **permutation test** for classifier accuracy: shuffle labels 1000 times, re-train, and build a null distribution. What p-value does your real accuracy achieve?
- Add **Common Spatial Patterns (CSP)** as a spatial filter before feature extraction. Does it help with only 7 channels?